In [48]:
import os
import re
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import keras
import numpy as np
from huggingface_hub import login
from datasets import load_dataset
import seaborn as sns
import matplotlib.pyplot as plt
import dagshub
import mlflow
from dotenv import load_dotenv
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from tensorflow.keras.callbacks import EarlyStopping

In [49]:
# Load the dataset

df = pd.read_csv("datasets/Customer_Support_Tickets_Cleaned.csv")

In [50]:
# Only using English language tickets for BERT model training
#df = df[df['language'] == 'en']

In [103]:
df_english = df[df['language'] == 'en']
df_english['queue'].value_counts()

queue
Technical Support                  8149
Product Support                    5304
Customer Service                   4269
IT Support                         3333
Billing and Payments               2897
Returns and Exchanges              1402
Service Outages and Maintenance    1106
Sales and Pre-Sales                 843
Human Resources                     553
General Inquiry                     404
Name: count, dtype: int64

In [52]:
import re

def clean_text(text):
    # Remove literal \n and \n\n
    text = re.sub(r'\\n+', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['bert_input'] = df['body'].apply(clean_text)

In [53]:
df['bert_input']

0        Sehr geehrtes Support-Team, ich möchte einen g...
1        Dear Customer Support Team, I am writing to re...
2        Dear Customer Support Team, I hope this messag...
3        Dear Customer Support Team, I hope this messag...
4        Dear Support Team, I hope this message reaches...
                               ...                        
61758    I am facing integration problems with IFTTT Do...
61759    Sehr geehrte Kundenservice, ich möchte die Int...
61760    Hello Customer Support, I am inquiring about t...
61761    Die Qualität unserer digitalen Strategie-Bearb...
61762    Sehr geehrte Customer Support-Team, ich schrei...
Name: bert_input, Length: 61763, dtype: object

In [97]:
df[df['queue'] == 'Books & Literature/Fiction']

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department,bert_input
28588,Bestellungen von signierten Ausgaben: App stür...,"Support-Team,\n\nich habe ein kritisches Probl...",NaN,NaN,Books & Literature/Fiction,high,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,bestellungen von signierten ausgaben app stürz...,support team ich habe ein kritisches problem d...,bestellungen von signierten ausgaben app stürz...,Other,"Support-Team, ich habe ein kritisches Problem:..."
28756,Problem beim Zugriff auf gekaufte E-Books aufg...,"Sehr geehrtes Support-Team,\nich habe Probleme...",NaN,NaN,Books & Literature/Fiction,medium,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,problem beim zugriff auf gekaufte e books aufg...,sehr geehrtes support team ich habe probleme m...,problem beim zugriff auf gekaufte e books aufg...,Other,"Sehr geehrtes Support-Team, ich habe Probleme ..."
28766,Problem beim Zugriff auf E-Books auf Blinkist,"Hallo, ich schreibe Ihnen, um ein Problem mit ...",NaN,NaN,Books & Literature/Fiction,low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,problem beim zugriff auf e books auf blinkist,hallo ich schreibe ihnen um ein problem mit me...,problem beim zugriff auf e books auf blinkist ...,Other,"Hallo, ich schreibe Ihnen, um ein Problem mit ..."
28809,Anfrage bezüglich Abweichungen im Tourneeplan ...,"Hallo Support,\n\nMir ist ein Terminkonflikt b...",NaN,NaN,Books & Literature/Fiction,low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,anfrage bezüglich abweichungen im tourneeplan ...,hallo support mir ist ein terminkonflikt bezüg...,anfrage bezüglich abweichungen im tourneeplan ...,Other,"Hallo Support, Mir ist ein Terminkonflikt bezü..."
28824,Problem beim Zugriff auf E-Book im Fiction Mic...,"Sehr geehrte Damen und Herren,\n\nich melde mi...",NaN,NaN,Books & Literature/Fiction,low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,problem beim zugriff auf e book im fiction mic...,sehr geehrte damen und herren ich melde mich u...,problem beim zugriff auf e book im fiction mic...,Other,"Sehr geehrte Damen und Herren, ich melde mich,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41613,Problem beim Zugriff auf E-Books gemeldet,"Hallo Support-Team,<br><br>ich schreibe Ihnen,...",NaN,NaN,Books & Literature/Fiction,high,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,problem beim zugriff auf e books gemeldet,hallo support team br br ich schreibe ihnen um...,problem beim zugriff auf e books gemeldet hall...,Other,"Hallo Support-Team,<br><br>ich schreibe Ihnen,..."
41653,Problem beim Zugriff auf E-Book: DRM-Fehler,"Hallo Support-Team,\n\nich melde ein Problem, ...",NaN,NaN,Books & Literature/Fiction,medium,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,problem beim zugriff auf e book drm fehler,hallo support team ich melde ein problem das i...,problem beim zugriff auf e book drm fehler hal...,Other,"Hallo Support-Team, ich melde ein Problem, das..."
41668,Probleme mit Apple Books DRM,"Support-Team,\nich wende mich an Sie wegen ein...",NaN,NaN,Books & Literature/Fiction,high,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,probleme mit apple books drm,support team ich wende mich an sie wegen eines...,probleme mit apple books drm support team ich ...,Other,"Support-Team, ich wende mich an Sie wegen eine..."
41680,Anfrage bezüglich des Archivs historischer Les...,"Support-Team,\nich schreibe Ihnen, um Zugang z...",NaN,NaN,Books & Literature/Fiction,very_low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,anfrage bezüglich des archivs historischer les...,support team ich schreibe ihnen um zugang zum ...,anfrage bezüglich des archivs historischer les...,Other,"Support-Team, ich schreibe Ihnen, um Zugang zu..."


In [80]:
(df['queue'].value_counts())

queue
Technical Support                         14186
Product Support                            8958
Customer Service                           7420
IT Support                                 5725
Billing and Payments                       4874
Returns and Exchanges                      2438
Service Outages and Maintenance            1912
Sales and Pre-Sales                        1490
Human Resources                             914
General Inquiry                             668
Pets & Animals/Pet Services                 386
News                                        383
IT & Technology/Security Operations         365
Autos & Vehicles/Sales                      364
Health/Medical Services                     362
Home & Garden/Home Improvement              361
Pets & Animals/Veterinary Care              356
Health/Mental Health                        347
Business & Industrial/Manufacturing         346
Online Communities/Forums                   343
Shopping/E-commerce               

In [ ]:
# Sales
df['queue'].isin(['Sales and Pre-Sales', 'Autos & Vehicles/Sales'])

In [ ]:
# Customer Success
df[df['queue'].isin(['Customer Service', 'Jobs & Education/Online Courses'])]

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department,bert_input
15,Inquiry for In-Depth Details on Financial Inst...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your interest in our financial p...,Request,Customer Service,medium,en,51.0,Inquiry,Product,...,Customer Support,NaN,NaN,NaN,NaN,inquiry for in depth details on financial inst...,dear customer support team i hope this message...,inquiry for in depth details on financial inst...,Customer Success,"Dear Customer Support Team, I hope this messag..."
17,Inquiry for Comprehensive Marketing Service De...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your interest in our marketing s...,Request,Customer Service,medium,en,51.0,Marketing,Feedback,...,Lead,NaN,NaN,NaN,NaN,inquiry for comprehensive marketing service de...,dear customer support team i hope this message...,inquiry for comprehensive marketing service de...,Customer Success,"Dear Customer Support Team, I hope this messag..."
29,Intermittent Access Problems on SaaS Platform,Currently facing sporadic connectivity difficu...,We appreciate you bringing this to our attenti...,Incident,Customer Service,medium,en,51.0,Network,Performance,...,Disruption,Tech Support,NaN,NaN,NaN,intermittent access problems on saas platform,currently facing sporadic connectivity difficu...,intermittent access problems on saas platform ...,Customer Success,Currently facing sporadic connectivity difficu...
33,Connectivity Problems with Smart Device,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for reaching out regarding the conne...,Incident,Customer Service,low,en,51.0,Network,Disruption,...,Tech Support,NaN,NaN,NaN,NaN,connectivity problems with smart device,dear customer support team i hope this message...,connectivity problems with smart device dear c...,Customer Success,"Dear Customer Support Team, I hope this messag..."
42,Alert for Unauthorized Entry,"Dear Customer Support,\n\nI have recently been...",We appreciate your reaching out regarding the ...,Problem,Customer Service,low,en,51.0,Security,IT,...,Alert,NaN,NaN,NaN,NaN,alert for unauthorized entry,dear customer support i have recently been not...,alert for unauthorized entry dear customer sup...,Customer Success,"Dear Customer Support, I have recently been no..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61730,Investment Analytics System abrupten Falles me...,"Sehr geehrte Kundenservice, ich möchte mich be...","Sehr geehrte <name>, vielen Dank für die Meldu...",Problem,Customer Service,medium,de,NaN,Technical,Bug,...,Resolution,Customer,NaN,NaN,NaN,investment analytics system abrupten falles me...,sehr geehrte kundenservice ich möchte mich bes...,investment analytics system abrupten falles me...,Customer Success,"Sehr geehrte Kundenservice, ich möchte mich be..."
61734,System Specifications for Action-Kamera Compat...,"Hello Customer Support, I am writing in regard...","Hello <name>, we appreciate your inquiry about...",Request,Customer Service,low,en,NaN,Technical,Customer,...,Guidance,Documentation,Communication,NaN,NaN,system specifications for action kamera compat...,hello customer support i am writing in regard ...,system specifications for action kamera compat...,Customer Success,"Hello Customer Support, I am writing in regard..."
61736,Benachrichtigung über unautorisierten Zugriff ...,Derzeit begegnen unautorisierten Zugriffsversu...,Es wurde eine E-Mail mit dem Thema 'unautorisi...,Problem,Customer Service,medium,de,NaN,Security,Virus,...,Tech Support,Feedback,NaN,NaN,NaN,benachrichtigung über unautorisierten zugriff ...,derzeit begegnen unautorisierten zugriffsversu...,benachrichtigung über unautorisierten zugriff ...,Customer Success,Derzeit begegnen unautorisierten Zugriffsversu...
61741,Investment Returns Discrepancy,I am contacting you to report a variance in th...,Your

In [ ]:
# Legal
df[df['queue'].isin(['Law & Government/Legal Advice', 'Law & Government/Government Services'])]

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department,bert_input
28607,Dringend: Kritisches Problem mit Compliance-Be...,"Sehr geehrte Damen und Herren vom Support,\n\n...",NaN,NaN,Law & Government/Legal Advice,high,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,dringend kritisches problem mit compliance ber...,sehr geehrte damen und herren vom support ich ...,dringend kritisches problem mit compliance ber...,Legal,"Sehr geehrte Damen und Herren vom Support, ich..."
28665,Bericht bezüglich eines Problems mit der DMV-T...,"Hallo Support-Team,\nich schreibe, um eine Klä...",NaN,NaN,Law & Government/Government Services,low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,bericht bezüglich eines problems mit der dmv t...,hallo support team ich schreibe um eine klärun...,bericht bezüglich eines problems mit der dmv t...,Legal,"Hallo Support-Team, ich schreibe, um eine Klär..."
28696,Frage zur Einhaltung der Bestimmungen für Auto...,"Hallo Support-Team,\n\nich schreibe Ihnen, um ...",NaN,NaN,Law & Government/Legal Advice,very_low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,frage zur einhaltung der bestimmungen für auto...,hallo support team ich schreibe ihnen um eine ...,frage zur einhaltung der bestimmungen für auto...,Legal,"Hallo Support-Team, ich schreibe Ihnen, um ein..."
28701,Formatierungsinkonsistenz in Vorlagen für Auto...,"Ich kontaktiere das Support-Team, um ein klein...",NaN,NaN,Law & Government/Legal Advice,low,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,formatierungsinkonsistenz in vorlagen für auto...,ich kontaktiere das support team um ein kleine...,formatierungsinkonsistenz in vorlagen für auto...,Legal,"Ich kontaktiere das Support-Team, um ein klein..."
28729,Anfrage bezüglich Datenzugriff nach dem Freedo...,"Support-Team,\n\nich melde mich, um wiederkehr...",NaN,NaN,Law & Government/Government Services,medium,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,anfrage bezüglich datenzugriff nach dem freedo...,support team ich melde mich um wiederkehrende ...,anfrage bezüglich datenzugriff nach dem freedo...,Legal,"Support-Team, ich melde mich, um wiederkehrend..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41617,Anfrage bezüglich eines Antrags nach dem Geset...,"Sehr geehrtes Support-Team,\n\nich schreibe Ih...",NaN,NaN,Law & Government/Government Services,medium,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,anfrage bezüglich eines antrags nach dem geset...,sehr geehrtes support team ich schreibe ihnen ...,anfrage bezüglich eines antrags nach dem geset...,Legal,"Sehr geehrtes Support-Team, ich schreibe Ihnen..."
41630,Kritischer Ausfall des E-Portals gemeldet,"Sehr geehrtes Support-Team,\n\nich schreibe Ih...",NaN,NaN,Law & Government/Government Services,critical,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,kritischer ausfall des e portals gemeldet,sehr geehrtes support team ich schreibe ihnen ...,kritischer ausfall des e portals gemeldet sehr...,Legal,"Sehr geehrtes Support-Team, ich schreibe Ihnen..."
41644,Frage zur Einhaltung der OTC-Kennzeichnung,"Support-Team,\nich schreibe Ihnen, um Unterstü...",NaN,NaN,Law & Government/Legal Advice,medium,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,frage zur einhaltung der otc kennzeichnung,support team ich schreibe ihnen um unterstützu...,frage zur einhaltung der otc kennzeichnung sup...,Legal,"Support-Team, ich schreibe Ihnen, um Unterstüt..."
41674,VoterReg API ausgefallen: Sofortige Maßnahmen ...,"Sehr geehrte Support-Mitarbeiter,\n\nwir haben...",NaN,NaN,Law & Government/Government Services,high,de,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,voterreg api ausgefallen sofortige maßnahmen e...,sehr geehrte support mitarbeiter wir haben der...,voterreg api ausgefallen sofortige maßnahmen e...,Legal,"Sehr geehrte Support-Mitarbeiter, wir haben de..."


In [ ]:
# Human Resources
df[df['queue'] == 'Human Resources', 'Jobs & Education/Recruitment']

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department,bert_input
21,Unable to Access Office Applications,"Dear Customer Support,\n\nWe are encountering ...",Thank you for providing a detailed explanation...,Incident,Human Resources,high,en,51.0,Account,Security,...,Tech Support,NaN,NaN,NaN,NaN,unable to access office applications,dear customer support we are encountering a pr...,unable to access office applications dear cust...,HR,"Dear Customer Support, We are encountering a p..."
93,Inquiry for Details of Recent Incident,"Dear Customer Support Team,\n\nI am reaching o...",Thank you for your email requesting details ab...,Incident,Human Resources,medium,en,51.0,Account,Incident,...,Feedback,NaN,NaN,NaN,NaN,inquiry for details of recent incident,dear customer support team i am reaching out t...,inquiry for details of recent incident dear cu...,HR,"Dear Customer Support Team, I am reaching out ..."
156,Urgent Login Problem with QuickBooks Account,"Dear Customer Support Team,\n\nI am reaching o...",Thank you for reaching out regarding your Quic...,Incident,Human Resources,medium,en,51.0,Account,Login,...,Tech Support,NaN,NaN,NaN,NaN,urgent login problem with quickbooks account,dear customer support team i am reaching out t...,urgent login problem with quickbooks account d...,HR,"Dear Customer Support Team, I am reaching out ..."
179,Peripheral Device Problems,"Dear Customer Support,\n\nI am submitting a re...",Thank you for reporting these peripheral devic...,Problem,Human Resources,low,en,51.0,Hardware,Performance,...,Tech Support,NaN,NaN,NaN,NaN,peripheral device problems,dear customer support i am submitting a report...,peripheral device problems dear customer suppo...,HR,"Dear Customer Support, I am submitting a repor..."
410,Incident SOC- DevSecOps Monitoring Alert,"Sehr geehrtes Support-Team,\n\nwir befinden un...",Vielen Dank für Ihre Nachricht. Bitte stellen ...,Incident,Human Resources,low,de,51.0,Security,Monitoring,...,Alert,Tech Support,NaN,NaN,NaN,incident soc devsecops monitoring alert,sehr geehrtes support team wir befinden uns de...,incident soc devsecops monitoring alert sehr g...,HR,"Sehr geehrtes Support-Team, wir befinden uns d..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61558,Enquiry About Customizable Features in SaaS Pr...,Could you provide details on the customization...,"Dear [Name], thank you for your interest in ou...",Request,Human Resources,low,en,NaN,Feature,Documentation,...,IT,Tech Support,NaN,NaN,NaN,enquiry about customizable features in saas pr...,could you provide details on the customization...,enquiry about customizable features in saas pr...,HR,Could you provide details on the customization...
61602,NaN,"Customer Support, I am reporting an issue with...",I apologize for the inconvenience caused by th...,Incident,Human Resources,medium,en,NaN,Bug,Crash,...,IT,Tech Support,NaN,NaN,NaN,none,customer support i am reporting an issue with ...,none customer support i am reporting an issue ...,HR,"Customer Support, I am reporting an issue with..."
61674,Unerwarteter Anstieg der Markebeneigungsmaßstä...,Es wurde ein Anstieg der Markebeneigungsmaßstä...,"Sehr geehrte/r [Name],\n\nich habe Ihre E-Mail...",Incident,Human Resources,medium,de,NaN,Performance,Feedback,...,NaN,NaN,NaN,NaN,NaN,unerwarteter anstieg der markebeneigungsmaßstä...,es wurde ein anstieg der markebeneigungsmaßstä...,unerwarteter anstieg der markebeneigungsmaßstä...,HR,Es wurde ein Anstieg der Markebeneigungsmaßstä...
61719,Concern About Data Breach in Hospital Systems,A data breach has been identified in the hospi...,We acknowledge receipt of your email concernin...,Problem,Human Resources,low,en,NaN,Security,IT,...,Training,Alert,NaN,NaN,NaN,concern about data breach in hospital systems,a data breach has been identified in the hospi...,concern about data breach in hospital systems ...

In [ ]:
# Technical
df[df['queue'].isin(['Technical Support', 'IT Support', 'IT & Technology/Software Development', 'IT & Technology/Network Infrastructure', 'IT & Technology/Security Operations', 'Online Communities/Social Networks', 'Online Communities/Forums', 'Autos & Vehicles/Maintenance', 'IT & Technology/Hardware Support'])]

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,...,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department,bert_input
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,...,Data Breach,NaN,NaN,NaN,NaN,wesentlicher sicherheitsvorfall,sehr geehrtes support team ich möchte einen gr...,wesentlicher sicherheitsvorfall sehr geehrtes ...,Technical,"Sehr geehrtes Support-Team, ich möchte einen g..."
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,...,IT,Tech Support,NaN,NaN,NaN,account disruption,dear customer support team i am writing to rep...,account disruption dear customer support team ...,Technical,"Dear Customer Support Team, I am writing to re..."
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,...,Feedback,NaN,NaN,NaN,NaN,feature query,dear customer support i hope this message reac...,feature query dear customer support i hope thi...,Technical,"Dear Customer Support, I hope this message rea..."
7,Connectivity Problems with Printer on MacBook Pro,"Dear Support Team,\n\nI am reporting a recurri...",Thank you for reaching out regarding the conne...,Incident,Technical Support,medium,en,51.0,Network,Hardware,...,Bug,Compatibility,NaN,NaN,NaN,connectivity problems with printer on macbook pro,dear support team i am reporting a recurring i...,connectivity problems with printer on macbook ...,Technical,"Dear Support Team, I am reporting a recurring ..."
8,Anfrage nach detaillierten Angaben zur Systema...,"Sehr geehrtes Kundensupport-Team,\n\nich hoffe...",Vielen Dank für Ihre Anfrage. Wir stellen Ihne...,Request,Technical Support,low,de,51.0,Documentation,Feedback,...,Tech Support,NaN,NaN,NaN,NaN,anfrage nach detaillierten angaben zur systema...,sehr geehrtes kundensupport team ich hoffe die...,anfrage nach detaillierten angaben zur systema...,Technical,"Sehr geehrtes Kundensupport-Team, ich hoffe, d..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61754,Investment Modeling Results Are Inaccurate,Detailed issue summary: The investment modelin...,"Dear <name>, I appreciate you bringing to my a...",Incident,Technical Support,medium,en,NaN,Technical,Bug,...,Documentation,Feedback,NaN,NaN,NaN,investment modeling results are inaccurate,detailed issue summary the investment modeling...,investment modeling results are inaccurate det...,Technical,Detailed issue summary: The investment modelin...
61755,Guidelines for Securing Medical Data in OBS St...,Seeking details on securing medical data using...,Offering general security guidelines for OBS S...,Request,Technical Support,high,en,NaN,Security,Documentation,...,NaN,NaN,NaN,NaN,NaN,guidelines for securing medical data in obs st...,seeking details on securing medical data using...,guidelines for securing medical data in obs st...,Technical,Seeking details on securing medical data using...
61757,Support for Marketing Enhancements,Request for assistance in improving digital ma...,Ready to help with your marketing support need...,Change,Technical Support,high,en,NaN,Feedback,Sales,...,Documentation,Tech Support,NaN,NaN,NaN,support for marketing enhancements,request for assistance in improving digital ma...,support for marketing enhancements request for...,Technical,Request for assistance in improving digital ma...
61758,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,I would be happy to assist with the IFTTT Dock...,Problem,Technical Support,low,en,NaN,Integration,Disruption,...,IT,Tech Support,NaN,NaN,NaN,assistance needed for ifttt docker integration,i am facing 

In [46]:
df = df[['bert_input', 'department']]

In [11]:


X = df['clean_sub_body']
le = LabelEncoder()

y = le.fit_transform(df['department'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

In [12]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", trainable=True)

In [13]:
# BERT

bert_tags = {
    "framework": "tensorflow.keras",
    "model_type": "BERT"
}

bert_train_params = {
    "optimizer": "adam",
    "loss": "sparse_categorical_crossentropy",
    "metrics": [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ],
    "epochs": 10,
    "batch_size": 32
}

bert_model_params = {
    "dropout": 0.1,
    "num_classes": len(le.classes_)
}

def create_bert_model(model_params):

    # Input Layer
    text_input = tf.keras.layers.Input(
        shape=(),
        dtype=tf.string,
        name="text"
    )

    # BERT preprocessing and encoder
    preprocessed_text = bert_preprocess(text_input)
    outputs = bert_encoder(preprocessed_text)

    # Classification Head
    x = tf.keras.layers.Dropout(
        model_params["dropout"],
        name="dropout"
    )(outputs["pooled_output"])

    output = tf.keras.layers.Dense(
        model_params["num_classes"],
        activation="softmax",
        name="output"
    )(x)

    model = tf.keras.Model(
        inputs=text_input,
        outputs=output
    )

    return model

bert_model = create_bert_model(bert_model_params)

In [14]:
models = [
    (
        "BERT",
        bert_model,
        (X_train, y_train),
        (X_test, y_test),
        bert_model_params,
        bert_train_params,
        bert_tags
    )
]

In [15]:
# ---------------------------------------------
# ROC Curve Function
# ---------------------------------------------

def create_roc_curve(model, model_name, X_test, y_test, labels):
    
    n_classes = len(labels)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)

    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    elif isinstance(model, tf.keras.Model):
        y_score = model.predict(X_test)

    else:
        raise ValueError(
            f"{type(model).__name__} does not support "
            "predict_proba or decision_function."
        )

    # ======================
    # Binary Classification
    # ======================
    if n_classes == 2:

        if y_score.ndim > 1:
            y_score_binary = y_score[:, 1]
        else:
            y_score_binary = y_score

        fpr, tpr, _ = roc_curve(
            y_test,
            y_score_binary
        )

        roc_auc = auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(
            fpr,
            tpr,
            lw=2,
            label=f"AUC = {roc_auc:.3f}"
        )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Multiclass Classification
    # ======================
    else:

        # One-hot encode y_test
        y_test_bin = label_binarize(
            y_test,
            classes=np.arange(n_classes)
        )

        roc_auc = roc_auc_score(
            y_test_bin,
            y_score,
            multi_class="ovr",
            average="weighted"
        )

        fig, ax = plt.subplots(figsize=(8, 6))

        for i in range(n_classes):

            fpr, tpr, _ = roc_curve(
                y_test_bin[:, i],
                y_score[:, i]
            )

            class_auc = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                lw=1.5,
                label=f"{labels[i]} (AUC={class_auc:.3f})"
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Common Plot Formatting
    # ======================
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")

    ax.legend(
        loc="lower right",
        fontsize=8
    )

    plt.tight_layout()

    # Save plot
    save_dir = os.path.join(
        "Images",
        "ROC_Curves"
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}.png"
    )

    fig.savefig(
        save_path,
        dpi=150
    )

    plt.close(fig)

    print(
        f"ROC Curve saved → {save_path} "
        f"(AUC: {roc_auc:.3f})"
    )

    return save_path, roc_auc

# ---------------------------------------------
# Confusion Matrix Function
# ---------------------------------------------

def create_confusion_matrix(model_name, y_true, y_pred, class_names, path="Images"):
    
    cm = confusion_matrix(y_true, y_pred)

    save_dir = os.path.join(path, "Confusion Matrix")
    os.makedirs(save_dir, exist_ok=True)

    safe_name = model_name.replace(" ", "_")
    save_path = os.path.join(save_dir, f"{safe_name}.png")

    plt.figure(figsize=(8, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"Confusion Matrix saved → {save_path}")

    return save_path

In [16]:
model_reports = []
model_history = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, model_params, train_params, tags in models:

    X_train, Y_train = train_set
    X_test, Y_test = test_set
    

    # Model Training
    model.compile(
        optimizer=train_params['optimizer'],
        loss=train_params['loss'],
        metrics=train_params['metrics']
    )


    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        min_delta=0.001
    )


    history = model.fit(
        X_train,
        Y_train,
        epochs=train_params['epochs'],
        batch_size=train_params['batch_size'],
        callbacks=[early_stop]
    )
    model_history.append(history)


    # Model Evaluation
    predictions = model.predict(X_test)
    y_pred = np.argmax(predictions, axis=1)


    # Saving Keras Model Locally
    model_save_dir = "saved_models"
    os.makedirs(model_save_dir, exist_ok=True)
    model_save_path = os.path.join(model_save_dir, f"{model_name}.keras")
    model.save(model_save_path)

    print(f"{model_name} saved to {model_save_path}")


    # Classification Report
    labels = le.classes_
    report = classification_report(
        Y_test,
        y_pred,
        target_names=labels,
        output_dict= True
    )
    model_reports.append(report)


    # ROC Curve
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, Y_test, labels)
    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    print(f"{model_name} ROC AUC saved successfully!")


    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=Y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)
    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Epoch 1/10
707/707 [==============================] - ETA: 0s - loss: 1.1307 - accuracy: 0.6263WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 386s 529ms/step - loss: 1.1307 - accuracy: 0.6263
Epoch 2/10
707/707 [==============================] - ETA: 0s - loss: 1.0961 - accuracy: 0.6330WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 375s 530ms/step - loss: 1.0961 - accuracy: 0.6330
Epoch 3/10
707/707 [==============================] - ETA: 0s - loss: 1.0874 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 373s 527ms/step - loss: 1.0874 - accuracy: 0.6331
Epoch 4/10
707/707 [==============================] - ETA: 0s - loss: 1.0854 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 373s 528ms/step - loss: 1.0854 - accuracy: 0.6331
Epoch 5/10
707/707 [==============================] - ETA: 0s - loss: 1.0846 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


...

707/707 [==============================] - 370s 524ms/step - loss: 1.0823 - accuracy: 0.6331
Epoch 9/10
707/707 [==============================] - ETA: 0s - loss: 1.1576 - accuracy: 0.6234WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 371s 525ms/step - loss: 1.1576 - accuracy: 0.6234
Epoch 10/10
707/707 [==============================] - ETA: 0s - loss: 1.1259 - accuracy: 0.6327WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


177/177 [==============================] - 41s 229ms/step
BERT saved to saved_models\BERT.keras
  1/177 [..............................] - ETA: 21s

c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


177/177 [==============================] - 41s 232ms/step
ROC Curve saved → Images\ROC_Curves\BERT.png (AUC: 0.506)
BERT ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\BERT.png
BERT Confusion Matrix saved successfully!
--------------------------------------------------------


In [18]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, Y_train = element[2]
    X_test, Y_test = element[3]
    model_params = element[4]
    train_params = element[5]
    tags = element[6]

    model_report = model_reports[i]
    history = model_history[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    with mlflow.start_run(run_name=model_name):

        # ---------------------------------------------------
        # Logging Parameters and Tags
        # ---------------------------------------------------

        mlflow.set_tags(tags)

        mlflow.log_params(model_params)
        mlflow.log_params(train_params)

        print(f"{model_name} parameters and tags logged!")

        #-----------------------------------------------------
        # Model training and validation performance tracking
        #-----------------------------------------------------

        for epoch in range(len(history.history['loss'])):
            mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
            mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)

        print(f"{model_name} training and validation performance logged!")

        #------------------------------------------------------
        # Log Model
        #------------------------------------------------------

        model_path = os.path.join("saved_models", f"{model_name}.keras")

        if os.path.exists(model_path):
            keras_model = keras.models.load_model(
                model_path,
                custom_objects={'KerasLayer': hub.KerasLayer}
            )
            model_info = mlflow.keras.log_model(
                keras_model,
                artifact_path=model_name
            )
            print(f"{model_name} model logged!")

        else:
            print(f"{model_name} model file not found at {model_path}!")


        #------------------------------------------------------
        # Model testing performance tracking
        #------------------------------------------------------

        # Log best epoch metrics explicitly (for clean experiment table comparison)
        best_epoch = np.argmin(history.history['loss'])
        mlflow.log_metric("best_epoch", best_epoch)
        mlflow.log_metric("best_train_accuracy", history.history['accuracy'][best_epoch])

        # Link test metrics to the LoggedModel using model_id
        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    if metric_name != "support":
                        mlflow_metrics[f"{label}_{metric_name}"] = float(value)
            else:
                mlflow_metrics[label] = float(metrics)

        # Pass model_id to link metrics to the model in the Models tab
        mlflow.log_metrics(mlflow_metrics, model_id=model_info.model_id)

        print(f"{model_name} metrics logged")

        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(model_cm_path[i], artifact_path="plots/Confusion Matrix")

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

BERT parameters and tags logged!
BERT training and validation performance logged!


2026/06/03 15:34:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 15:34:16 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model\data\model\assets
2026/06/03 15:34:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model, flavor: tensorflow). Fall back to return ['tensorflow==2.10.1']. Set logging level to DEBUG to see the full traceback. 
2026/06/03 15:34:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


BERT model logged!
BERT metrics logged
Metrics Logged
🏃 View run BERT at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/9cd165ea087e4f1680883482a574e965
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


In [ ]:
"""# BERT layers
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
outputs = bert_encoder(preprocessed_text)

# Classification head
l = tf.keras.layers.Dropout(
    0.1,
    name="dropout"
)(outputs['pooled_output'])

l = tf.keras.layers.Dense(
    len(le.classes_),
    activation='softmax',
    name="output"
)(l)

# Model
bert_model = tf.keras.Model(
    inputs=text_input,
    outputs=l
)

METRICS = [
    tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')
]

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=METRICS
)

model.fit(X_train, y_train, epochs=10)"""

Epoch 1/10
1545/1545 [==============================] - 372s 236ms/step - loss: 1.1777 - accuracy: 0.5978
Epoch 2/10
1545/1545 [==============================] - 381s 246ms/step - loss: 1.1094 - accuracy: 0.6228
Epoch 3/10
1545/1545 [==============================] - 364s 236ms/step - loss: 1.0907 - accuracy: 0.6281
Epoch 4/10
1545/1545 [==============================] - 369s 239ms/step - loss: 1.0826 - accuracy: 0.6320
Epoch 5/10
1545/1545 [==============================] - 362s 234ms/step - loss: 1.0775 - accuracy: 0.6340
Epoch 6/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0698 - accuracy: 0.6388
Epoch 7/10
1545/1545 [==============================] - 361s 234ms/step - loss: 1.0674 - accuracy: 0.6381
Epoch 8/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0647 - accuracy: 0.6385
Epoch 9/10
1545/1545 [==============================] - 365s 236ms/step - loss: 1.0630 - accuracy: 0.6400
Epoch 10/10
1545/1545 [=======================